# Train OCR biển số Việt Nam trên Kaggle T4 x2

Notebook ổn định cho chạy dài: tự động tìm input, copy code vào `/kaggle/working`, ghi log ra file, hạn chế spam output, hỗ trợ resume từ `last.pt`, và zip kết quả cuối.

Thứ tự khuyến nghị: chạy từ trên xuống. Nếu chạy qua đêm, dùng `Save Version -> Save & Run All` để Kaggle lưu output.


In [ ]:
import os
import sys
import time
import shutil
import subprocess
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, display, HTML

os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::UserWarning"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORK_ROOT = Path("/kaggle/working")
LOG_DIR = WORK_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

FULL_EPOCHS = 280
MEDIUM_EPOCHS = 30


def tail_text(path, n=80):
    path = Path(path)
    if not path.exists():
        return ""
    return "".join(path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-n:])


def run_cmd(args, log_path=None, show_all=False):
    log_path = Path(log_path) if log_path else None
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::UserWarning"
    print("Running:", " ".join(map(str, args)))
    if log_path:
        print("Log:", log_path)
    with (log_path.open("w", encoding="utf-8") if log_path else open(os.devnull, "w")) as log:
        proc = subprocess.Popen(
            list(map(str, args)),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in proc.stdout:
            log.write(line)
            log.flush()
            text = line.strip()
            if show_all and text:
                print(text, flush=True)
            elif text and (
                text[0].isdigit()
                or text.startswith(("READY_", "Checking", "Charset:", "Rows:", "Forward OK", "Decoder:", "train:", "val:", "test:", "charset:"))
            ):
                print(text, flush=True)
        rc = proc.wait()
    if rc != 0:
        if log_path:
            print("Last log lines:")
            print(tail_text(log_path, 100))
        raise RuntimeError(f"Command failed with exit code {rc}")
    print("DONE")


def finished_epoch(out_dir):
    hist = Path(out_dir) / "history.csv"
    if not hist.exists():
        return 0
    df = pd.read_csv(hist)
    if df.empty:
        return 0
    return int(df["epoch"].max())


def run_train(out_dir, epochs, log_name, extra_args=None, resume=True):
    out_dir = Path(out_dir)
    done = finished_epoch(out_dir)
    if done >= epochs:
        print(f"Already finished {done}/{epochs} epochs:", out_dir)
        return
    args = [
        "torchrun", "--nproc_per_node=2", "/kaggle/working/kaggle_scratch_ocr/train_best.py",
        "--config", "/kaggle/working/kaggle_scratch_ocr/configs/t4x2_safe.yaml",
        "--data-root", "/kaggle/working/ocr_prepared",
        "--out-dir", str(out_dir),
        "--epochs", str(epochs),
    ]
    last = out_dir / "last.pt"
    if resume and last.exists() and done > 0:
        args += ["--resume", str(last)]
        print(f"Resume from epoch {done}:", last)
    if extra_args:
        args += list(map(str, extra_args))
    run_cmd(args, LOG_DIR / log_name)


def show_history(out_dir, rows=5):
    out_dir = Path(out_dir)
    hist = out_dir / "history.csv"
    print("Output:", out_dir)
    print("Files:", sorted(p.name for p in out_dir.iterdir()) if out_dir.exists() else [])
    if hist.exists():
        df = pd.read_csv(hist)
        display(df.tail(rows))
        print("Finished epoch:", int(df["epoch"].max()))
    else:
        print("No history.csv yet")


def zip_results(out_dir="/kaggle/working/scratch_plate_ocr_t4x2", zip_path="/kaggle/working/plate_ocr_result.zip"):
    out_dir = Path(out_dir)
    zip_path = Path(zip_path)
    important_files = [
        "best.pt",
        "best_ema.pt",
        "last.pt",
        "charset.txt",
        "config_used.json",
        "history.csv",
        "predictions_val.csv",
        "error_report.csv",
        "hard_examples.csv",
    ]
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for name in important_files:
            path = out_dir / name
            if path.exists():
                z.write(path, arcname=f"model/{name}")
        if LOG_DIR.exists():
            for path in LOG_DIR.glob("*.log"):
                z.write(path, arcname=f"logs/{path.name}")
    print("ZIP file created successfully at:", zip_path)
    print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))
    # Display alternative download guides
    display(HTML('''
    <div style="background-color: #e3f2fd; border-left: 5px solid #2196f3; padding: 15px; margin: 10px 0;">
        <h4 style="margin-top:0; color: #0d47a1;">💡 Hướng dẫn tải file ZIP kết quả:</h4>
        <p>Do Kaggle chặn tải trực tiếp bằng liên kết <code>FileLink</code> thông thường (gây lỗi 404), bạn hãy tải theo 1 trong 2 cách sau:</p>
        <ol>
            <li><b>Cách 1 (Khuyên dùng):</b> Nhìn sang <b>cột bên phải màn hình Kaggle</b>, ở mục <b>"Files" / "Output"</b>, tìm file <code>plate_ocr_result.zip</code>, di chuột qua và nhấn biểu tượng <b>Download (mũi tên tải xuống)</b>.</li>
            <li><b>Cách 2:</b> Nếu bạn chạy chế độ "Save Version -> Save & Run All", sau khi chạy xong, hãy vào trang Viewer của Notebook và tải trong tab <b>"Output"</b> ở bên phải.</li>
        </ol>
    </div>
    '''))
    display(FileLink(str(zip_path)))


## 1. Kiểm tra GPU và input


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

print("\n/kaggle/input:")
for p in Path("/kaggle/input").iterdir():
    print(" ", p)


## 2. Dò dataset và copy code vào working

Cell này chịu được cả hai kiểu Kaggle mount: trực tiếp `/kaggle/input/<dataset>` hoặc nằm sâu dưới `/kaggle/input/datasets/...`.


In [ ]:
INPUT_ROOT = Path("/kaggle/input")

data_candidates = [
    Path("/kaggle/input/vietnam-plate-ocr-data/ocr_dataset"),
    Path("/kaggle/input/vietnam-plate-ocr-data"),
]
code_candidates = [
    Path("/kaggle/input/kaggle-scratch-ocr"),
    Path("/kaggle/input/kaggle-scratch-ocr/kaggle_scratch_ocr"),
]

data_candidates += [p.parent for p in INPUT_ROOT.rglob("train_labels.txt")]
code_candidates += [p.parent for p in INPUT_ROOT.rglob("train_best.py")]

DATA_ROOT = next((p for p in data_candidates if (p / "train_labels.txt").exists()), None)
CODE_ROOT = next((p for p in code_candidates if (p / "train_best.py").exists()), None)
WORK_CODE = Path("/kaggle/working/kaggle_scratch_ocr")

print("Detected DATA_ROOT:", DATA_ROOT)
print("Detected CODE_ROOT:", CODE_ROOT)

if DATA_ROOT is None:
    raise FileNotFoundError("Không tìm thấy train_labels.txt trong dataset ảnh. Kiểm tra lại Add Data.")
if CODE_ROOT is None:
    raise FileNotFoundError("Không tìm thấy train_best.py trong dataset code. Kiểm tra lại Add Data.")

if WORK_CODE.exists():
    shutil.rmtree(WORK_CODE)
shutil.copytree(CODE_ROOT, WORK_CODE)

print("Copied code to:", WORK_CODE)
print("Data files:", sorted(p.name for p in DATA_ROOT.iterdir()))
print("Code files:", sorted(p.name for p in WORK_CODE.iterdir()))


## 3. Chuẩn bị dữ liệu


In [ ]:
run_cmd([
    "python", "/kaggle/working/kaggle_scratch_ocr/prepare_dataset.py",
    "--data-root", str(DATA_ROOT),
    "--out-root", "/kaggle/working/ocr_prepared",
], LOG_DIR / "prepare_dataset.log")

PREPARED = Path("/kaggle/working/ocr_prepared")
print((PREPARED / "summary.txt").read_text(encoding="utf-8"))
train_df = pd.read_csv(PREPARED / "train.csv")
val_df = pd.read_csv(PREPARED / "val.csv")
display(train_df.head())
print("train:", len(train_df), "val:", len(val_df))


## 4. Ready check

Phải thấy `READY_CHECK_PASS` rồi mới train.


In [ ]:
run_cmd([
    "python", "/kaggle/working/kaggle_scratch_ocr/check_ready.py",
    "--data-root", "/kaggle/working/ocr_prepared",
    "--multi-sizes", "40x160,48x192,56x224",
    "--img-h", "48",
    "--img-w", "192",
], LOG_DIR / "ready_check.log")


## 5. Sanity run 3 epoch

Chạy nhanh để kiểm tra pipeline tạo checkpoint/log/output. Nếu cell này lỗi thì không chạy tiếp.

> **Lưu ý:** `--synthetic-ratio 0` để tắt sinh ảnh tổng hợp trong sanity, giúp chạy nhanh hơn.


In [ ]:
run_train(
    out_dir="/kaggle/working/sanity_plate_ocr_t4x2",
    epochs=3,
    log_name="sanity.log",
    extra_args=["--train-limit", "512", "--val-limit", "256", "--batch-size", "64", "--synthetic-ratio", "0"],
    resume=True,
)
show_history("/kaggle/working/sanity_plate_ocr_t4x2", rows=10)


## 6. Medium run 30 epoch

Bước này giúp kiểm tra loss/accuracy có cải thiện trước khi chạy full.


In [ ]:
run_train(
    out_dir="/kaggle/working/medium_plate_ocr_t4x2",
    epochs=MEDIUM_EPOCHS,
    log_name="medium.log",
    resume=True,
)
show_history("/kaggle/working/medium_plate_ocr_t4x2", rows=10)

MEDIUM = Path("/kaggle/working/medium_plate_ocr_t4x2")
pred_path = MEDIUM / "predictions_val.csv"
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    display(pred.head(30))
    print("Raw accuracy:", pred["correct"].mean())
    print("Rule accuracy:", pred["rule_correct"].mean())
err = MEDIUM / "error_report.csv"
if err.exists():
    display(pd.read_csv(err).head(30))


## 7. Full run 280 epoch

Cell này tự resume nếu đã có `/kaggle/working/scratch_plate_ocr_t4x2/last.pt`. Khi chạy qua đêm, nên dùng `Save Version -> Save & Run All`.


In [ ]:
# Warm-start: copy medium best.pt to scratch dir if scratch has no checkpoint
import shutil
medium_best = Path("/kaggle/working/medium_plate_ocr_t4x2/best.pt")
scratch_last = Path("/kaggle/working/scratch_plate_ocr_t4x2/last.pt")
scratch_dir = Path("/kaggle/working/scratch_plate_ocr_t4x2")
if medium_best.exists() and not scratch_last.exists():
    scratch_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(medium_best, scratch_last)
    # Copy charset and config too
    for f in ["charset.txt", "config_used.json"]:
        src = Path("/kaggle/working/medium_plate_ocr_t4x2") / f
        if src.exists():
            shutil.copy(src, scratch_dir / f)
    print("Warm-started from medium best.pt")
else:
    print("No warm-start needed (scratch already has checkpoint or medium not found)")

run_train(
    out_dir="/kaggle/working/scratch_plate_ocr_t4x2",
    epochs=FULL_EPOCHS,
    log_name="full.log",
    resume=True,
)
show_history("/kaggle/working/scratch_plate_ocr_t4x2", rows=10)
zip_results()


## 8. Xem kết quả full run


In [ ]:
OUT_DIR = Path("/kaggle/working/scratch_plate_ocr_t4x2")
show_history(OUT_DIR, rows=20)

pred_path = OUT_DIR / "predictions_val.csv"
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    display(pred.head(30))
    print("Raw accuracy:", pred["correct"].mean())
    print("Rule accuracy:", pred["rule_correct"].mean())

err = OUT_DIR / "error_report.csv"
if err.exists():
    display(pd.read_csv(err).head(30))


## 9. Inference thử một ảnh


In [ ]:
test_images = sorted((DATA_ROOT / "test").glob("*.jpg")) + sorted((DATA_ROOT / "test").glob("*.png"))
if not test_images:
    print("Không tìm thấy ảnh test .jpg/.png")
else:
    image = test_images[0]
    print("Image:", image)
    run_cmd([
        "python", "/kaggle/working/kaggle_scratch_ocr/infer.py",
        "--checkpoint", "/kaggle/working/scratch_plate_ocr_t4x2/best.pt",
        "--image", str(image),
    ], LOG_DIR / "infer.log", show_all=True)


## 10. Zip kết quả thủ công

Có thể chạy lại cell này bất cứ lúc nào để tạo zip mới từ checkpoint/log hiện có.


In [ ]:
zip_results()


## 11. Tải trực tiếp file kết quả (.zip)

Cell này giúp hiển thị link tải trực tiếp các file `.zip` kết quả từ thư mục làm việc.

In [ ]:
import os
from IPython.display import FileLink, display

# Tìm tất cả file .zip đang có thực tế trong thư mục làm việc
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if not zip_files:
    print("Chưa tìm thấy file zip nào trong thư mục /kaggle/working.")
    print("Vui lòng đợi tiến trình train chạy xong hoặc chạy cell nén kết quả trước.")
else:
    print("Danh sách link tải trực tiếp từ Kaggle (Click vào tên file bên dưới để tải):")
    for f in zip_files:
        display(FileLink(f))
